# Song Lyrics Generator using LSTM
venv(3.13.5)(Python 3.13.5)
### Project Overview
This project builds an LSTM model that learns patterns from song lyrics and generates new lyrics word by word.

**Steps followed:**
1. Import libraries
2. Load & preprocess data
3. Train / test split
4. Build model
5. Compile
6. Callbacks
7. Train
8. Generate lyrics & save

## Step 1 — Import libraries

In [50]:
import numpy as np
import pandas as pd
import pickle

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

from sklearn.model_selection import train_test_split

print('All libraries imported successfully')

All libraries imported successfully


## Step 2 — Load & preprocess data

```

In [51]:
artist_name = "drake"   # change this anytime

file_path = f"D:\\generative AI\\RNN\\LSTM+RNN\\LSTM RNN\\archive\\{artist_name}.txt"

with open(file_path, 'r', encoding='utf-8') as f:
    all_text = f.read()

all_lyrics = all_text.lower()

print(f"Loaded {artist_name} dataset ✅")

Loaded drake dataset ✅


In [ ]:

# Use text from TXT files
all_lyrics = all_text.lower()

# Optional: reduce size (VERY IMPORTANT ⚡)
all_lyrics = all_lyrics[:500000]

print(f'Total characters: {len(all_lyrics)}')

Total characters: 199038


In [ ]:

tokenizer = Tokenizer()
tokenizer.fit_on_texts([all_lyrics])
total_words = len(tokenizer.word_index) + 1
print(f'Vocabulary size: {total_words}')

Vocabulary size: 3789


In [ ]:

input_sequences = []

for line in all_lyrics.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

print(f'Total sequences: {len(input_sequences)}')

Total sequences: 35680


In [ ]:
# Pad sequences
max_sequence_len = max([len(x) for x in input_sequences])
print(f'Max sequence length: {max_sequence_len}')

input_sequences = np.array(pad_sequences(
    input_sequences,
    maxlen=max_sequence_len,
    padding='pre'
))

# Split into X (input) and y (next word label)
X, y = input_sequences[:, :-1], input_sequences[:, -1]


print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

Max sequence length: 92
X shape: (35680, 91)
y shape: (35680,)


## Step 3 — Train / test split

In [56]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (28544, 91), Test: (7136, 91)


## Step 4 — Build model

In [ ]:

model = Sequential()

# Embedding layer — converts word index → dense vector
model.add(Embedding(total_words,100,input_length=max_sequence_len - 1))
# embedding dimension

model.add(LSTM(200, return_sequences=True))
model.add(Dropout(0.2))

model.add(LSTM(150))
model.add(Dropout(0.2))

# Output layer — one neuron per word in vocabulary
model.add(Dense(total_words, activation='softmax'))

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_6 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_7 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## Step 5 — Compile

In [ ]:

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
print('Model compiled')

Model compiled


## Step 6 — Callbacks

In [59]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,                        # wait 5 epochs before stopping
    restore_best_weights=True
)
print('EarlyStopping callback ready')

EarlyStopping callback ready


## Step 7 — Train

In [60]:
history = model.fit(
    X_train, y_train,
    epochs=2,
    batch_size=64,                     # larger batch = faster training
    validation_data=(X_test, y_test),
    verbose=1,
    callbacks=[early_stopping]
)

Epoch 1/2
 78/446 ━━━━━━━━━━━━━━━━━━━━ 1:00 165ms/step - accuracy: 0.0327 - loss: 7.3199

KeyboardInterrupt: 

## Step 8 — Generate lyrics & save



In [ ]:
def sample_with_temperature(preds, temperature=1.0):
    """
    Instead of always picking the top word (argmax),
    we sample randomly but weighted by probability.
    temperature controls how creative/random the output is.
    """
    preds = np.log(preds + 1e-10) / temperature   # scale by temperature
    preds = np.exp(preds)
    preds = preds / np.sum(preds)                  # re-normalize to sum=1
    return np.random.choice(len(preds), p=preds)   # sample from distribution


def generate_lyrics(seed_text, next_words=50, temperature=1.0):
    """
    seed_text  : starting words (e.g. 'i knew you were')
    next_words : how many words to generate
    temperature: creativity level (0.5 = safe, 1.5 = creative)
    """
    result = seed_text

    for _ in range(next_words):
        # Tokenize current text
        token_list = tokenizer.texts_to_sequences([result])[0]

        # Pad to required input length
        token_list = pad_sequences(
            [token_list],
            maxlen=max_sequence_len - 1,
            padding='pre'
        )

        # Get word probabilities from model
        preds = model.predict(token_list, verbose=0)[0]

        # Sample next word using temperature
        predicted_index = sample_with_temperature(preds, temperature)

        # Convert index back to word
        output_word = ''
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break

        result += ' ' + output_word

    return result

In [ ]:
# Try generating lyrics with different temperatures
seed = 'i knew you were'

print('=== Temperature 0.5 (safe) ===')
print(generate_lyrics(seed, next_words=40, temperature=0.5))

print('\n=== Temperature 1.0 (balanced) ===')
print(generate_lyrics(seed, next_words=40, temperature=1.0))

print('\n=== Temperature 1.5 (creative) ===')
print(generate_lyrics(seed, next_words=40, temperature=1.5))

=== Temperature 0.5 (safe) ===
i knew you were it my way of the day you got i can were for the time and it down oh to no no no want you go you be i know you know i don't know me i will know you be

=== Temperature 1.0 (balanced) ===
i knew you were cried she's than my hands i think up and apart to think i say doe every middle with slips all he's thursday together beautiful we see you can done to come falling you never they're enough i'm dude just into

=== Temperature 1.5 (creative) ===
i knew you were top broken figure known you fighting he missed love oh home that wishing she i feeds of kissin' around they say you direction uh can mine better justin young without talking rushes boy in his tu armando vacant kiss we'll


In [ ]:
# Save model and tokenizer
model.save('song_lyrics_lstm.h5')
print('Model saved as song_lyrics_lstm.h5')

with open('lyrics_tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)
print('Tokenizer saved as lyrics_tokenizer.pickle')

Model saved as song_lyrics_lstm.h5
Tokenizer saved as lyrics_tokenizer.pickle
